# Part 4 — Vector Databases: Embeddings & Semantic Similarity

This notebook demonstrates semantic embeddings using `sentence-transformers`. 10 sentences across 3 topics (Cricket, Cooking, Cybersecurity) are embedded using `all-MiniLM-L6-v2`, a 10×10 cosine similarity heatmap is rendered, and the top-2 most similar sentences are retrieved for the query: *"The bowler took three wickets in one over"*.

In [1]:
!pip install sentence-transformers seaborn matplotlib scikit-learn -q
print('All packages ready.')

Installing packages...


In [2]:
from sentence_transformers import SentenceTransformer
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics.pairwise import cosine_similarity

model = SentenceTransformer('all-MiniLM-L6-v2')
print('Model loaded: all-MiniLM-L6-v2')

Model loaded: all-MiniLM-L6-v2


In [3]:
sentences = [
    # Cricket (indices 0-3)
    'The batsman hit a magnificent century in the final over.',
    'India won the Test series after outstanding bowling.',
    'The fielder took a stunning catch near the boundary.',
    'Rain interrupted play on the third day of the Test match.',
    # Cooking (indices 4-6)
    'Sauté the onions in olive oil until they turn golden brown.',
    'Add a pinch of salt and simmer the curry on low heat.',
    'Knead the dough gently and let it rest before baking.',
    # Cybersecurity (indices 7-9)
    'The hacker exploited a zero-day vulnerability in the web app.',
    'Multi-factor authentication reduces the risk of account breach.',
    'Ransomware encrypted the hospital records, demanding Bitcoin.',
]

labels = [
    'Cricket-1','Cricket-2','Cricket-3','Cricket-4',
    'Cooking-1','Cooking-2','Cooking-3',
    'CyberSec-1','CyberSec-2','CyberSec-3'
]
topics = ['Cricket']*4 + ['Cooking']*3 + ['CyberSec']*3

print(f'Total sentences : {len(sentences)}')
print('Topics          : Cricket (4), Cooking (3), Cybersecurity (3)\n')
for i, (s, t) in enumerate(zip(sentences, topics)):
    print(f' {i:2d}  [{t:<10}]  {s}')

Total sentences : 10
Topics          : Cricket (4), Cooking (3), Cybersecurity (3)

 0  [Cricket   ]  The batsman hit a magnificent century in the final over.
 1  [Cricket   ]  India won the Test series after outstanding bowling.
 2  [Cricket   ]  The fielder took a stunning catch near the boundary.
 3  [Cricket   ]  Rain interrupted play on the third day of the Test match.
 4  [Cooking   ]  Sauté the onions in olive oil until they turn golden brown.
 5  [Cooking   ]  Add a pinch of salt and simmer the curry on low heat.
 6  [Cooking   ]  Knead the dough gently and let it rest before baking.
 7  [CyberSec  ]  The hacker exploited a zero-day vulnerability in the web app.
 8  [CyberSec  ]  Multi-factor authentication reduces the risk of account breach.
 9  [CyberSec  ]  Ransomware encrypted the hospital records, demanding Bitcoin.


In [4]:
embeddings = model.encode(sentences, convert_to_numpy=True)
print(f'Embedding matrix shape: {embeddings.shape}')

Embedding matrix shape: (10, 384)


In [5]:
sim_matrix = cosine_similarity(embeddings)

plt.figure(figsize=(10, 8))
sns.heatmap(
    sim_matrix,
    xticklabels=labels,
    yticklabels=labels,
    annot=True,
    fmt='.2f',
    cmap='YlOrRd',
    vmin=0,
    vmax=1,
    linewidths=0.5,
    square=True
)
plt.title('10×10 Cosine Similarity Matrix\n(Cricket | Cooking | Cybersecurity)', fontsize=13, pad=14)
plt.xticks(rotation=45, ha='right', fontsize=9)
plt.yticks(rotation=0, fontsize=9)
plt.tight_layout()
plt.savefig('cosine_similarity_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('Heatmap saved as cosine_similarity_heatmap.png')

Heatmap saved as cosine_similarity_heatmap.png


In [6]:
query = 'The bowler took three wickets in one over'
query_embedding = model.encode([query], convert_to_numpy=True)
query_sims = cosine_similarity(query_embedding, embeddings)[0]
top2_idx = np.argsort(query_sims)[::-1][:2]

print(f"Query: '{query}'\n")
print('Top 2 most similar sentences:\n')
for rank, idx in enumerate(top2_idx, start=1):
    print(f'Rank {rank}')
    print(f'  Similarity : {query_sims[idx]:.4f}')
    print(f'  Label      : {labels[idx]}')
    print(f'  Sentence   : {sentences[idx]}\n')

Query: 'The bowler took three wickets in one over'

Top 2 most similar sentences:

Rank 1
  Similarity : 0.6284
  Label      : Cricket-2
  Sentence   : India won the Test series after outstanding bowling.

Rank 2
  Similarity : 0.5731
  Label      : Cricket-3
  Sentence   : The fielder took a stunning catch near the boundary.
